### MetaboFM Gradio app

In [ ]:
# vqa_app.py — MSI viewer + single-question CLS Q/A
# pip install gradio==4.* pandas scikit-learn timm transformers torch torchvision torchaudio

# --- Windows event-loop fix (MUST be before importing gradio) ---
import os, asyncio
if os.name == "nt":
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    except Exception:
        pass

import re, json, math, hashlib, glob
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import gradio as gr

import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================
# Heads we expose (CLS-only)
# =========================
CLS_HEADS = [
    "organism",
    "polarity",
    "organ",
    "condition",
    "analyzerType",
    "ionisationSource",
]

SEED = 6740
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Fixed question (as trained)
# =========================
FIXED_QUESTION = "what is the organism?"

# ===========================================
# Image helpers (preview + robust normalization)
# ===========================================
def _safe_uint8(img: np.ndarray) -> np.ndarray:
    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(img, [1, 99])
    if hi <= lo: hi = lo + 1e-6
    img = np.clip((img - lo) / (hi - lo), 0, 1)
    return (img * 255).astype(np.uint8)

def _rgb_from_patch_pca(patch: np.ndarray) -> np.ndarray:
    # patch: (C,H,W) in [0,1]
    C, H, W = patch.shape
    X = patch.reshape(C, -1).T  # (H*W, C)
    if C > 0:
        X = X - X.mean(axis=0, keepdims=True)
    comps = min(3, max(1, C))
    if C >= comps:
        pca = PCA(n_components=comps, svd_solver="randomized")
        Y = pca.fit_transform(X)
    else:
        Y = np.zeros((H * W, comps), dtype=np.float32)
    rgb = np.zeros((H * W, 3), dtype=np.float32)
    rgb[:, :comps] = Y
    rgb = rgb.reshape(H, W, 3)
    out = np.zeros_like(rgb, dtype=np.uint8)
    for i in range(3):
        out[..., i] = _safe_uint8(rgb[..., i])
    return out

def _single_channel_gray(patch: np.ndarray, idx: int) -> np.ndarray:
    idx = int(np.clip(int(idx), 0, max(0, patch.shape[0] - 1)))
    ch = patch[idx]
    return np.stack([_safe_uint8(ch)] * 3, axis=-1)

def _load_npz_patch(npz_file) -> Tuple[np.ndarray, np.ndarray, str]:
    path = npz_file.name if hasattr(npz_file, "name") else npz_file
    with np.load(path, mmap_mode="r") as z:
        patch = z["patch"].astype(np.float32)  # (C,H,W)
        mz    = z["mz"].astype(np.float32)
    # normalize if uint16 scale
    if patch.max() > 1.0:
        patch /= 65535.0
    return patch, mz, path

# ===========================================
# Intent detection (CLS focus; legacy fallbacks)
# ===========================================
INTENT = {
    "organism":         re.compile(r"\b(organism|species)\b", re.I),
    "polarity":         re.compile(r"\bpolari(?:ty)?\b", re.I),
    "organ":            re.compile(r"\b(organ(?:ism)?\s*part|organ\b|tissue)\b", re.I),
    "condition":        re.compile(r"\b(condition|status)\b", re.I),
    "analyzerType":     re.compile(r"\banaly[sz]er(?:\s*type)?\b", re.I),
    "ionisationSource": re.compile(r"\bion(i[sz]ation)?\s*source\b|\bion[i|z]iser\b", re.I),
    # legacy examples
    "left_right": re.compile(r"\b(left|right).*(bright|darker|brighter)\b", re.I),
    "count_5pct": re.compile(r"\bhow many\b.*(five|5)\s*percent.*(non[- ]?zero|nonzero)", re.I),
    "mz_yesno":   re.compile(r"\b(ion|peak).*(near|around)\s*m/?z\s*([0-9]+(?:\.[0-9]+)?)", re.I),
}

def detect_intent(question: str) -> Tuple[str, Optional[str]]:
    q = (question or "").strip()
    if not q:
        return ("none", None)
    for head in CLS_HEADS:
        if INTENT[head].search(q):
            return ("cls", head)
    if INTENT["left_right"].search(q):
        return ("legacy_left_right", None)
    if INTENT["count_5pct"].search(q):
        return ("legacy_count_5pct", None)
    m = INTENT["mz_yesno"].search(q)
    if m:
        return ("legacy_mz_yesno", float(m.group(3)))
    return ("cls", "auto")

# ===========================================
# Summarization helpers
# ===========================================
def _best_cls_head(cls_dict: dict):
    best_h, best_v = None, None
    for h, d in (cls_dict or {}).items():
        if not isinstance(d, dict):
            continue
        if best_v is None or float(d.get("confidence", 0.0)) > float(best_v.get("confidence", 0.0)):
            best_h, best_v = h, d
    return best_h, best_v

def _cls_summary(cls_dict: dict, target: Optional[str]):
    if not cls_dict:
        return "I couldn't infer a class from this model."
    if target and target != "auto":
        for k, d in cls_dict.items():
            if k.lower() == target.lower():
                return f"**{k}** → **{d.get('pred','?')}**."
    k, d = _best_cls_head(cls_dict)
    if k is None or d is None:
        return "I couldn't infer a class from this model."
    return f"**{k}** → **{d.get('pred','?')}**."

def summarize_filtered(result_item: dict, intent_kind: str, intent_target: Optional[str]):
    r = result_item.get("result", {})
    cls_dict = r.get("cls")
    if intent_kind == "cls":
        return _cls_summary(cls_dict, intent_target)
    yn = r.get("yesno", None)
    if intent_kind in ("legacy_left_right", "legacy_mz_yesno", "legacy_count_5pct"):
        if isinstance(yn, dict) and "pred" in yn:
            return f"**{str(yn['pred']).upper()}**."
        return _cls_summary(cls_dict, target=None)
    return _cls_summary(cls_dict, target=None)

# ===========================================
# Model components (match training/eval)
# ===========================================
import timm
from transformers import AutoModel, AutoTokenizer

class FrozenBackbone(nn.Module):
    """timm ViT backbone returning [B, D] CLS-like embedding; all params frozen."""
    def __init__(self, timm_name: str, pretrained: bool = True):
        super().__init__()
        self.m = timm.create_model(timm_name, pretrained=pretrained, num_classes=0)
        for p in self.m.parameters():
            p.requires_grad_(False)
        self.m.eval()

    @torch.no_grad()
    def forward(self, x3):  # [N,3,H,W]
        feats = self.m.forward_features(x3)
        if isinstance(feats, dict):
            if 'x_norm_clstoken' in feats:  return feats['x_norm_clstoken']
            if 'cls_token' in feats:        return feats['cls_token']
            if 'avgpool' in feats:          return feats['avgpool']
            for k in ('last_hidden_state', 'tokens', 'x'):
                if k in feats and torch.is_tensor(feats[k]):
                    t = feats[k]
                    return t[:, 0] if t.dim() == 3 else t
        if torch.is_tensor(feats):
            return feats[:, 0] if feats.dim() == 3 else feats
        return feats.mean(dim=-2)

def crop_resize_to_target(x3: torch.Tensor, target=224, patch_multiple=16) -> torch.Tensor:
    _, _, H, W = x3.shape
    Hc = (H // patch_multiple) * patch_multiple
    Wc = (W // patch_multiple) * patch_multiple
    dh = (H - Hc) // 2; dw = (W - Wc) // 2
    if Hc > 0 and Wc > 0:
        x3 = x3[:, :, dh:dh+Hc, dw:dw+Wc]
    if (Hc, Wc) != (target, target):
        x3 = F.interpolate(x3, size=(target, target), mode="bilinear", align_corners=False)
    return x3

class HFTextEnc(nn.Module):
    """Trainable HuggingFace encoder + linear projection (used during training)."""
    def __init__(self, name: str, out_dim: int):
        super().__init__()
        self.model = AutoModel.from_pretrained(name)
        hid = self.model.config.hidden_size
        self.proj = nn.Linear(hid, out_dim)
        self.name = name

    def forward(self, ids, mask):
        out = self.model(input_ids=ids, attention_mask=mask, return_dict=True)
        x = (out.last_hidden_state * mask.unsqueeze(-1)).sum(1) / (mask.sum(1, keepdim=True) + 1e-6)
        return self.proj(x)  # [B, out_dim]

class Fusion(nn.Module):
    """Exactly matches training/eval: LN(img), LN(txt), MLP on concat."""
    def __init__(self, d_img, d_txt, d_out):
        super().__init__()
        self.ln_img = nn.LayerNorm(d_img)
        self.ln_txt = nn.LayerNorm(d_txt)
        self.mlp = nn.Sequential(
            nn.Linear(d_img + d_txt, d_out),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_out, d_out),
        )
    def forward(self, zi, zt):
        x = torch.cat([self.ln_img(zi), self.ln_txt(zt)], dim=-1)
        return self.mlp(x)

class VQAHeads(nn.Module):
    """Per-task linear heads over fused embedding z_fused."""
    def __init__(self, embed_dim: int, cls_spaces: Dict[str, List[str]]):
        super().__init__()
        self.heads = nn.ModuleDict({k: nn.Linear(embed_dim, len(v)) for k, v in cls_spaces.items()})
        for lin in self.heads.values():
            nn.init.trunc_normal_(lin.weight, std=0.02); nn.init.zeros_(lin.bias)

    def forward(self, z):  # [B, D_fused]
        return {k: h(z) for k, h in self.heads.items()}

# ===========================================
# Loading & inference utils
# ===========================================
RUN_ROOT = Path("vqa")
IMG_CACHE_DIR = RUN_ROOT / "_img_cache"
STATS_DIR    = RUN_ROOT / "_stats_cache"

def _pick_ckpt(run_dir: str) -> str:
    best = os.path.join(run_dir, "best.pt")
    last = os.path.join(run_dir, "last.pt")
    if os.path.exists(best): return best
    if os.path.exists(last): return last
    cand = [os.path.join(run_dir, f) for f in os.listdir(run_dir) if f.endswith(".pt")]
    if not cand:
        raise FileNotFoundError(f"No checkpoint found in {run_dir}")
    return sorted(cand)[-1]

def _load_config(run_dir: str) -> dict:
    cfg_path = os.path.join(run_dir, "config.json")
    if not os.path.exists(cfg_path):
        raise FileNotFoundError(f"Missing config.json in {run_dir}")
    return json.load(open(cfg_path, "r"))

def _per_image_channel_zscore(x: torch.Tensor) -> torch.Tensor:
    # x: [1,C,H,W], return z-scored per-channel over H*W (fallback of last resort)
    B, C, H, W = x.shape
    xm = x.view(B, C, -1).mean(dim=-1, keepdim=True).view(B, C, 1, 1)
    xs = x.view(B, C, -1).std(dim=-1, keepdim=True).view(B, C, 1, 1).clamp_min(1e-6)
    return (x - xm) / xs

def _embed_key(path, timm_id, patch_multiple, target_size):
    h = hashlib.sha1(f"{timm_id}|{patch_multiple}|{target_size}|{path}".encode()).hexdigest()
    return IMG_CACHE_DIR / f"{h}.npy"

def _load_stats_or_none(channels_per_view: int, input_size: int):
    """
    Try to load a mu/std cache that matches channels_per_view & input_size.
    We pick the most recent matching file from STATS_DIR.
    """
    pattern = str(STATS_DIR / f"mu_std_c{int(channels_per_view)}_in{int(input_size)}_*.npz")
    files = sorted(glob.glob(pattern))
    if not files:
        return None, None
    z = np.load(files[-1], allow_pickle=True)
    mu, std = z["mu"], z["std"]
    return mu.astype(np.float32), np.maximum(std.astype(np.float32), 1e-6)

def _select_k_first(patch: np.ndarray, k: int) -> np.ndarray:
    """
    Match training behavior: keep channels in original order, truncate/pad to k.
    If C >= k: take first k. If C < k: repeat from start to reach k.
    """
    C, H, W = patch.shape
    if C >= k:
        return patch[:k]
    reps = int(np.ceil(k / max(C, 1)))
    tiled = np.tile(patch, (reps, 1, 1))[:k]
    return tiled

@torch.no_grad()
def _encode_image_mean_cls_fallback_with_stats(
    patch_chw: np.ndarray,
    bb: FrozenBackbone,
    *,
    target: int,
    patch_multiple: int,
    channels_per_view: int,
    input_size: int,
    mu: Optional[np.ndarray],
    std: Optional[np.ndarray],
    channels_per_step: int = 16
) -> torch.Tensor:
    """
    Fallback encoder that attempts to match training:
    - select top-variance channels -> exactly channels_per_view
    - normalize with dataset mu/std if available; else per-image z-score
    - crop/resize & mean-CLS over channels
    """
    # 1) channel selection
    x = _select_k_first(patch_chw, channels_per_view)  # [K,H,W]
    x = torch.from_numpy(x).unsqueeze(0)  # [1,K,H,W]

    # 2) normalization
    if mu is not None and std is not None and len(mu) >= channels_per_view and len(std) >= channels_per_view:
        mu_t = torch.from_numpy(mu[:channels_per_view]).view(1, -1, 1, 1)
        sd_t = torch.from_numpy(std[:channels_per_view]).view(1, -1, 1, 1)
        x = (x - mu_t) / sd_t
    else:
        x = _per_image_channel_zscore(x)

    B, K, H, W = x.shape

    # 3) flatten channels to batch, replicate to 3ch
    x_flat = x.permute(0, 2, 3, 1).contiguous().view(B * K, 1, H, W)
    x_rgb  = x_flat.repeat(1, 3, 1, 1)

    # 4) chunk over channels
    cls_chunks = []
    step = channels_per_step if target == 224 else max(4, channels_per_step // 2)
    for s in range(0, B * K, step):
        e = min(s + step, B * K)
        xr = crop_resize_to_target(x_rgb[s:e], target=target, patch_multiple=patch_multiple)
        cls = bb(xr.to(DEVICE))             # [N, D_img]
        cls_chunks.append(cls.float().cpu())
    cls_all = torch.cat(cls_chunks, dim=0)  # [K, D_img]
    z_img = cls_all.mean(dim=0, keepdim=True)  # [1, D_img]
    return F.normalize(z_img, dim=-1)

def _softmax_top(logits: torch.Tensor, vocab: List[str]) -> Tuple[str, float]:
    probs = logits.softmax(dim=-1)[0]  # [K]
    conf, idx = float(probs.max().item()), int(probs.argmax().item())
    pred = vocab[idx] if 0 <= idx < len(vocab) else "unknown"
    return pred, conf

def _encode_fixed_question(tok, text_enc, max_len):
    t = tok(FIXED_QUESTION, padding=False, truncation=True, max_length=max_len, return_tensors="pt")
    ids  = t["input_ids"].to(DEVICE)
    mask = t["attention_mask"].to(DEVICE)
    with torch.no_grad():
        z_txt = F.normalize(text_enc(ids, mask), dim=-1)
    return z_txt

def _normalize_question(q: str) -> str:
    q = (q or "").strip()
    if not q:
        return "What organism is this sample?"
    return q

def _apply_unknown_penalty(logits: torch.Tensor, vocab: List[str], penalty: float) -> torch.Tensor:
    """
    Subtract a constant from the 'unknown' logit (if present) to reduce its dominance.
    logits: [1, K]
    """
    if penalty <= 0:
        return logits
    try:
        unk_idx = vocab.index("unknown")
    except ValueError:
        return logits
    out = logits.clone()
    out[0, unk_idx] = out[0, unk_idx] - float(penalty)
    return out

# ===========================================
# Core: run one question on one .npz
# ===========================================
def vqa_on_npz_single(
    run_dir: str,
    npz_path: str,
    question: str
) -> Dict:
    """
    Loads ckpt + config from run_dir (frozen image backbone setup), runs one question on one .npz.
    Returns:
      { "result": { "cls": { head: {"pred": str, "confidence": float}, ... } },
        "meta": { "used_cached_embedding": bool, "used_dataset_stats": bool } }
    """
    # ----- Load cfg + ckpt
    ckpt_path = _pick_ckpt(run_dir)
    cfg = _load_config(run_dir)
    state = torch.load(ckpt_path, map_location="cpu")

    # Backbone config from training
    timm_id        = cfg.get("timm_id", "vit_small_patch14_dinov2.lvd142m")
    patch_multiple = int(cfg.get("patch_multiple", 14))
    target_size    = int(cfg.get("target_size", 518))
    text_model_id  = cfg.get("hf_text_model", "sentence-transformers/all-MiniLM-L6-v2")
    text_out_dim   = int(cfg.get("text_out_dim", cfg.get("embed_dim_text", 384)))
    channels_per_view = int(cfg.get("channels_per_view", 64))
    input_size     = int(cfg.get("input_size", 256))

    # Class spaces (from ckpt if available to ensure exact vocab)
    cls_spaces = state.get("cls_spaces", cfg.get("cls_spaces", {h: ["unknown"] for h in CLS_HEADS}))

    # ----- Build modules
    bb = FrozenBackbone(timm_id, pretrained=True).to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(text_model_id)
    text_enc = HFTextEnc(text_model_id, out_dim=text_out_dim).to(DEVICE).eval()

    d_img     = int(cfg.get("embed_dim_image", 384))
    fused_dim = int(cfg.get("embed_dim_fused", d_img + text_out_dim))
    fusion    = Fusion(d_img, text_out_dim, fused_dim).to(DEVICE).eval()
    heads     = VQAHeads(embed_dim=fused_dim, cls_spaces=cls_spaces).to(DEVICE).eval()

    # ----- Load weights (strict for fusion/heads, relaxed for text_enc)
    if "text_enc" in state:
        text_enc.load_state_dict(state["text_enc"], strict=False)
    if "fusion" in state:
        fusion.load_state_dict(state["fusion"], strict=True)
    if "heads" in state:
        heads.load_state_dict(state["heads"], strict=True)

    # ----- Load sample
    with np.load(npz_path, mmap_mode="r") as z:
        patch = z["patch"].astype(np.float32)  # (C,H,W)
        if patch.max() > 1.0: patch /= 65535.0

    # ----- Prefer cached image embedding (identical to eval); fallback to train-like path
    embed_path = _embed_key(npz_path, timm_id, patch_multiple, target_size)
    meta_used_cache = False
    meta_used_stats = False

    if embed_path.exists():
        z_img = torch.from_numpy(np.load(embed_path, mmap_mode="r")).unsqueeze(0).to(DEVICE).float()
        z_img = F.normalize(z_img, dim=-1)
        meta_used_cache = True
    else:
        mu, std = _load_stats_or_none(channels_per_view=channels_per_view, input_size=input_size)
        meta_used_stats = mu is not None and std is not None
        z_img = _encode_image_mean_cls_fallback_with_stats(
            patch, bb,
            target=target_size,
            patch_multiple=patch_multiple,
            channels_per_view=channels_per_view,
            input_size=input_size,
            mu=mu,
            std=std,
            channels_per_step=16
        ).to(DEVICE)

    # ----- Encode text as fixed prompt (matches training)
    _ = _normalize_question(question)  # user text ignored to match training; kept for future multi-prompt training
    z_txt = _encode_fixed_question(tok, text_enc, max_len=int(cfg.get("text_max_len", 64)))

    # ----- Fuse & predict
    with torch.no_grad():
        z_fused = fusion(z_img, z_txt)            # [1, D_fused]
        logits_dict = heads(z_fused)              # dict of head_name -> [1, K]

    UNKNOWN_LOGIT_PENALTY = 0.7  # try 0.5..1.0 if 'unknown' is still over-predicted

    # ----- Build result
    res = {"cls": {}, "meta": {"used_cached_embedding": bool(meta_used_cache),
                            "used_dataset_stats": bool(meta_used_stats)}}
    for head, vocab in cls_spaces.items():
        if head not in logits_dict:
            continue
        # apply penalty before softmax
        adj = _apply_unknown_penalty(logits_dict[head], vocab, UNKNOWN_LOGIT_PENALTY)
        pred, conf = _softmax_top(adj, vocab)
        res["cls"][head] = {"pred": pred, "confidence": conf}

    return {"result": res}

# ===========================================
# Preview handler (no models)
# ===========================================
def _preview(npz_file, view_mode, ch_index):
    if npz_file is None:
        return None
    patch, mz, _ = _load_npz_patch(npz_file)
    if view_mode == "PCA RGB":
        return _rgb_from_patch_pca(patch)
    return _single_channel_gray(patch, int(ch_index))

# ===========================================
# Core run handler (loads run, answers one question)
# ===========================================
def run_basic(vqa_run, npz_file, view_mode, ch_index, question, state):
    if npz_file is None:
        return None, "Please upload a sample first.", state

    # preview image
    patch, mz, real_path = _load_npz_patch(npz_file)
    if view_mode == "PCA RGB":
        rgb = _rgb_from_patch_pca(patch)
    else:
        rgb = _single_channel_gray(patch, int(ch_index))

    if not question or not str(question).strip():
        return rgb, "Type a question like **What organism is this sample?**", state

    try:
        out = vqa_on_npz_single(run_dir=str(vqa_run), npz_path=real_path, question=str(question))
        ikind, itarget = detect_intent(question)
        answer_text = summarize_filtered(out, ikind, itarget)

        meta = out.get("result", {}).get("meta", {})
        used_cache = meta.get("used_cached_embedding", False)
        used_stats = meta.get("used_dataset_stats", False)

    except Exception as e:
        answer_text = f"Error while running the model: {e}"

    return rgb, answer_text, state

# ===========================================
# UI (Gradio)
# ===========================================
with gr.Blocks(title="metaboFM") as demo:
    gr.Markdown(
        "## metaboFM\n"
        "Upload an MSI patch (`.npz` with arrays `patch` (C,H,W)), preview it, and ask:\n"
        "- *What organism is this sample?*\n"
        "- *What is the ionization polarity?*\n"
        "- *Which organ is this sample from?*\n"
        "- *What is the sample condition?*\n"
        "- *What analyzer type / ionisation source was used?*\n"
    )

    with gr.Row():
        with gr.Column(scale=2):
            npz_file  = gr.File(label="Upload .npz (must contain 'patch' and 'mz')", file_types=[".npz"])
            view_mode = gr.Radio(choices=["PCA RGB", "Single Channel"], value="PCA RGB", label="View")
            ch_index  = gr.Slider(label="Channel (for Single Channel view)", minimum=0, maximum=255, step=1, value=0)
            img_out   = gr.Image(label="Preview", type="numpy")

        with gr.Column(scale=1):
            gr.Image("metabofm.png", label="", show_label=False, container=False, interactive=False, height=220)

            question  = gr.Textbox(
                label="Ask a question",
                placeholder="e.g., What organism is this sample?",
            )
            with gr.Row():
                btn_org = gr.Button("What organism is this sample?")
                btn_pol = gr.Button("What is the ionization polarity?")
            with gr.Row():
                btn_orgn = gr.Button("Which organ is this sample from?")
                btn_cond = gr.Button("What is the sample condition?")
            with gr.Row():
                btn_an   = gr.Button("What analyzer type was used?")
                btn_ions = gr.Button("What ionisation source was used?")

            ask_btn   = gr.Button("Ask", variant="primary")
            answer    = gr.Markdown("")

            with gr.Accordion("Model run directory", open=False):
                vqa_run   = gr.Textbox(label="VQA run directory (contains best.pt/last.pt + config.json)", value="20251113_182023")

            state = gr.State(value=None)

    # Preview on change
    for ctrl in [npz_file, view_mode, ch_index]:
        ctrl.change(
            fn=_preview,
            inputs=[npz_file, view_mode, ch_index],
            outputs=img_out
        )

    # Quick-pick question helpers
    def _q_org():   return "What organism is this sample?"
    def _q_pol():   return "What is the ionization polarity?"
    def _q_orgn():  return "Which organ is this sample from?"
    def _q_cond():  return "What is the sample condition?"
    def _q_an():    return "What analyzer type was used?"
    def _q_ions():  return "What ionisation source was used?"

    btn_org.click(fn=_q_org, outputs=question).then(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state],
    )
    btn_pol.click(fn=_q_pol, outputs=question).then(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state],
    )
    btn_orgn.click(fn=_q_orgn, outputs=question).then(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state],
    )
    btn_cond.click(fn=_q_cond, outputs=question).then(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state],
    )
    btn_an.click(fn=_q_an, outputs=question).then(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state],
    )
    btn_ions.click(fn=_q_ions, outputs=question).then(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state],
    )

    # Ask
    ask_btn.click(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state]
    )
    question.submit(
        fn=run_basic,
        inputs=[vqa_run, npz_file, view_mode, ch_index, question, state],
        outputs=[img_out, answer, state]
    )

if __name__ == "__main__":
    # Gradio queue helps keep a single event loop path for uploads/progress on Windows
    demo.queue(status_update_rate=1)
    demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\eozturk7\AppData\Local\miniconda3\envs\magic\lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 403, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "c:\Users\eozturk7\AppData\Local\miniconda3\envs\magic\lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
  File "c:\Users\eozturk7\AppData\Local\miniconda3\envs\magic\lib\site-packages\fastapi\applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\eozturk7\AppData\Local\miniconda3\envs\magic\lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\eozturk7\AppData\Local\miniconda3\envs\magic\lib\site-packages\starlette\middleware\errors.py", line 186, in __call__
    raise exc
  File "c:\Users\eozturk7\Ap